In [ ]:
print("="*70)
print("ArbItro - Ensemble Test  (Pipeline 1 + Pipeline 2)")
print("="*70)

import tensorflow as tf
print(f"\nTensorFlow version: {tf.__version__}")

gpu_name = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if gpu_name:
    gpu_info = gpu_name[0].split(',')
    print(f"  GPU  : {gpu_info[0].strip()}")
    print(f"  VRAM : {gpu_info[1].strip()}")
    print("\n  System Resources:")
    !free -h | grep Mem
    !nvidia-smi --query-gpu=memory.free,memory.total --format=csv
else:
    print("\n  No GPU found")

print("\n" + "="*70)

In [ ]:
import os, sys

try:
    from google.colab import drive
    USE_COLAB = True
    drive.mount('/content/drive')
    BASE_DIR        = '/content/drive/MyDrive/ArbItro'
    P2_CODE_DIR     = os.path.join(BASE_DIR, 'ArbItro_code')
    P1_CODE_DIR     = os.path.join(BASE_DIR, 'ArbItro_code', 'pipeline1')
    DATASET_DIR     = os.path.join(BASE_DIR, 'ArbItro_dataset', 'dataset')
    TRAINING_ROOT   = os.path.join(BASE_DIR, 'ArbItro_Training')
except ImportError:
    USE_COLAB = False
    BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '../..'))
    P2_CODE_DIR   = os.path.join(BASE_DIR, 'model', 'src', 'pipeline2')
    P1_CODE_DIR   = os.path.join(BASE_DIR, 'model', 'src', 'pipeline1')
    DATASET_DIR   = os.path.join(BASE_DIR, 'dataset')
    TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')

sys.path.insert(0, P2_CODE_DIR)
sys.path.insert(1, P1_CODE_DIR)

print(f"  USE_COLAB    : {USE_COLAB}")
print(f"  P2_CODE_DIR  : {P2_CODE_DIR}")
print(f"  P1_CODE_DIR  : {P1_CODE_DIR}")
print(f"  DATASET_DIR  : {DATASET_DIR}")
print(f"  TRAINING_ROOT: {TRAINING_ROOT}")

required_paths = [
    os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    os.path.join(DATASET_DIR, 'test'),
]
print("\nVerifying test paths:")
for path in required_paths:
    status = "Found" if os.path.exists(path) else "Not found"
    print(f"  {status}: {path}")

In [ ]:
!pip install opencv-python-headless

In [ ]:
import shutil

WORK_DIR = '/content/arbitro_model' if USE_COLAB else os.path.join(BASE_DIR, '.arbitro_work')
os.makedirs(WORK_DIR, exist_ok=True)

shutil.copy(os.path.join(P2_CODE_DIR, 'data_loader.py'), os.path.join(WORK_DIR, 'data_loader_p2.py'))
shutil.copy(os.path.join(P2_CODE_DIR, 'model.py'),       os.path.join(WORK_DIR, 'model_p2.py'))
shutil.copy(os.path.join(P1_CODE_DIR, 'data_loader.py'), os.path.join(WORK_DIR, 'data_loader_p1.py'))
shutil.copy(os.path.join(P1_CODE_DIR, 'model.py'),       os.path.join(WORK_DIR, 'model_p1.py'))

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"  Working dir: {WORK_DIR}")
for f in sorted(os.listdir(WORK_DIR)):
    print(f"  {f}")

In [ ]:
try:
    import data_loader_p2 as _dl_p2
    import model_p2       as _m_p2
    Pipe2Generator                = _dl_p2.ArbItroDataGenerator
    BinaryBalancedAccuracy_P2     = _m_p2.BinaryBalancedAccuracy
    MulticlassBalancedAccuracy_P2 = _m_p2.MulticlassBalancedAccuracy
    print("  Pipeline 2 imports successful!")
except Exception as e:
    print(f"  Import error (Pipeline 2): {e}")
    raise

try:
    import data_loader_p1 as _dl_p1
    import model_p1       as _m_p1
    Pipe1Generator                = _dl_p1.ArbItroDataGenerator
    BinaryBalancedAccuracy_P1     = _m_p1.BinaryBalancedAccuracy
    MulticlassBalancedAccuracy_P1 = _m_p1.MulticlassBalancedAccuracy
    print("  Pipeline 1 imports successful!")
except Exception as e:
    print(f"  Import error (Pipeline 1): {e}")
    raise

In [ ]:
TEST_CONFIG = {
    'json_path'        : os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    'video_dir'        : os.path.join(DATASET_DIR, 'test'),
    'pipe2_model_path' : os.path.join(TRAINING_ROOT, 'models', 'pipeline2.keras'),
    'pipe1_model_path' : os.path.join(TRAINING_ROOT, 'models', 'pipeline1.keras'),
    'dim'              : (224, 398),
    'n_frames'         : 16,
    'max_clips'        : 4,
    'batch_size'       : 28,
    'shuffle'          : False,
}

_OFF_MAP = {"No offence": 0, "": 0, "Between": 1, "Offence": 1}
_ACT_MAP = {
    "": 0, "Dont know": 0, "Dive": 0,
    "Challenge": 1, "Tackling": 1, "Standing tackling": 1,
    "Holding": 2, "Pushing": 2, "Elbowing": 2,
    "High leg": 3,
}
MAP_OFF = {0: "No Offence",   1: "Offence"}
MAP_SEV = {0: "No Card",      1: "Yellow",      2: "Red"}
MAP_ACT = {0: "Neutral/Dive", 1: "Tackles",     2: "Upper Body", 3: "High Leg"}

def _get_sev(row):
    try:
        v = float(row.get('Severity', 0))
        if v >= 5.0: return 2
        if v >= 3.0: return 1
        return 0
    except: return 0

for key in ('pipe2_model_path', 'pipe1_model_path'):
    status = "Found" if os.path.exists(TEST_CONFIG[key]) else "Not found"
    print(f"  {status}: {TEST_CONFIG[key]}")

In [ ]:
import traceback

try:
    test_gen_p2 = Pipe2Generator(
        json_path=TEST_CONFIG['json_path'],
        base_video_path=TEST_CONFIG['video_dir'],
        batch_size=TEST_CONFIG['batch_size'],
        max_clips=TEST_CONFIG['max_clips'],
        dim=TEST_CONFIG['dim'],
        n_frames=TEST_CONFIG['n_frames'],
        shuffle=False,
        use_auxiliary_features=True,
        augment=False,
    )
    print(f"  Pipeline 2 test samples: {test_gen_p2.n_samples}")
except Exception as e:
    print(f"  Generator error (Pipeline 2): {e}")
    traceback.print_exc()
    raise

def masked_mean(inputs):
    x, mask = inputs[0], inputs[1]
    mask = tf.expand_dims(mask, axis=-1)
    return tf.reduce_sum(x * mask, axis=1) / (tf.reduce_sum(mask, axis=1) + 1e-7)

try:
    model_p2 = tf.keras.models.load_model(
        TEST_CONFIG['pipe2_model_path'],
        custom_objects={'masked_mean': masked_mean},
        compile=False,
    )
    print(f"\n  Pipeline 2 model loaded : {TEST_CONFIG['pipe2_model_path']}")
    print(f"  Output heads            : {model_p2.output_names}")
except Exception as e:
    print(f"  Error loading Pipeline 2 model: {e}")
    traceback.print_exc()
    raise

In [ ]:
import numpy as np

print("Running inference (Pipeline 2)...")
preds_p2 = model_p2.predict(test_gen_p2, verbose=1)
pred_dict_p2 = {name: val for name, val in zip(model_p2.output_names, preds_p2)}
print(f"  Output keys: {list(pred_dict_p2.keys())}")

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix as sk_cm


def print_raw_report(y_true, y_pred, labels, title):
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    print(f"  Accuracy         : {acc:.2%}")
    print(f"  Balanced Accuracy: {bal_acc:.2%}")
    print("-" * 70)
    cm = sk_cm(y_true, y_pred, labels=list(labels.keys()))
    print(f"  {'CLASS':<22} | {'CORRECT / TOTAL':<18} | {'CLASS ACC':<10}")
    print("-" * 70)
    for idx, name in labels.items():
        if idx < len(cm):
            total = int(np.sum(cm[idx, :]))
            correct = int(cm[idx, idx])
            acc_c = correct / total if total > 0 else 0.0
            print(f"  {name:<22} | {correct}/{total:<17} | {acc_c:.2%}")
    print("-" * 70)
    return {'accuracy': float(acc), 'balanced_accuracy': float(bal_acc)}


def print_action_raw_report(test_samples, y_pred, raw_classes, act_map, title):
    y_true = np.array([act_map.get(s.get('Action class', ''), 0) for s in test_samples])
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)
    print(f"  Accuracy         : {acc:.2%}")
    print(f"  Balanced Accuracy: {bal_acc:.2%}")
    print("-" * 70)
    print(f"  {'CLASS':<22} | {'CORRECT / TOTAL':<18} | {'CLASS ACC':<10}")
    print("-" * 70)
    for cls_name in raw_classes:
        indices = [i for i, s in enumerate(test_samples) if s.get('Action class', '') == cls_name]
        target_grp = act_map.get(cls_name, 0)
        correct = sum(1 for i in indices if y_pred[i] == target_grp)
        total = len(indices)
        if total > 0:
            label = "N/A (Empty)" if cls_name == "" else cls_name
            print(f"  {label:<22} | {correct}/{total:<17} | {correct / total:.2%}")
    print("-" * 70)
    return {'accuracy': float(acc), 'balanced_accuracy': float(bal_acc)}


num_samples = pred_dict_p2[next(iter(pred_dict_p2))].shape[0]
test_samples = test_gen_p2.samples[:num_samples]

y_true_off_p2 = np.array([_OFF_MAP.get(s.get('Offence', ''), 0) for s in test_samples])
y_pred_off_p2 = (pred_dict_p2['head_offence'].flatten() > 0.5).astype(int)
print_raw_report(y_true_off_p2, y_pred_off_p2, MAP_OFF, "HEAD OFFENCE (Pipeline 2)")

y_pred_act_p2 = np.argmax(pred_dict_p2['head_action'], axis=1)
raw_act_classes = ["", "Dont know", "Challenge", "Tackling", "Standing tackling",
                   "High leg", "Holding", "Pushing", "Elbowing", "Dive"]
print_action_raw_report(test_samples, y_pred_act_p2, raw_act_classes, _ACT_MAP,
                        "HEAD ACTION (Pipeline 2 — Raw Classes)")

y_true_sev = np.array([_get_sev(s) for s in test_samples])
y_pred_sev_p2 = np.argmax(pred_dict_p2['head_severity'], axis=1)
print_raw_report(y_true_sev, y_pred_sev_p2, MAP_SEV, "HEAD SEVERITY (Pipeline 2)")

In [ ]:
try:
    test_gen_p1 = Pipe1Generator(
        json_path=TEST_CONFIG['json_path'],
        base_video_path=TEST_CONFIG['video_dir'],
        batch_size=TEST_CONFIG['batch_size'],
        dim=TEST_CONFIG['dim'],
        n_frames=TEST_CONFIG['n_frames'],
        shuffle=False,
        use_auxiliary_features=True,
        augment=False,
    )
    print(f"  Pipeline 1 test samples: {test_gen_p1.n_samples}")
except Exception as e:
    print(f"  Generator error (Pipeline 1): {e}")
    traceback.print_exc()
    raise

try:
    model_p1 = tf.keras.models.load_model(
        TEST_CONFIG['pipe1_model_path'],
        custom_objects={
            'BinaryBalancedAccuracy': BinaryBalancedAccuracy_P1,
            'MulticlassBalancedAccuracy': MulticlassBalancedAccuracy_P1,
        },
        compile=False,
    )
    print(f"\n  Pipeline 1 model loaded : {TEST_CONFIG['pipe1_model_path']}")
    print(f"  Output heads            : {model_p1.output_names}")
except Exception as e:
    print(f"  Error loading Pipeline 1 model: {e}")
    traceback.print_exc()
    raise

print("\nRunning inference (Pipeline 1)...")
preds_p1 = model_p1.predict(test_gen_p1, verbose=1)
pred_dict_p1 = {name: val for name, val in zip(model_p1.output_names, preds_p1)}
print(f"  Output keys: {list(pred_dict_p1.keys())}")

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


def safe_argmax(arr):
    arr = np.array(arr)
    if arr.ndim > 1 and arr.shape[1] > 1:
        return np.argmax(arr, axis=1)
    return np.round(arr).astype(int).flatten()


y_sev_p2 = safe_argmax(pred_dict_p2['head_severity'])
y_sev_p1 = safe_argmax(pred_dict_p1['head_severity'])

limit = min(len(y_sev_p2), len(y_sev_p1))
y_sev_p2 = y_sev_p2[:limit]
y_sev_p1 = y_sev_p1[:limit]

# Ensemble: base = Pipeline 1, Pipeline 2 overrides RED
y_ens = y_sev_p1.copy()
y_ens[y_sev_p2 == 2] = 2

changes = np.sum(y_ens != y_sev_p1)
print(f"  Samples evaluated        : {limit}")
print(f"  Pipeline 2 RED overrides : {changes}")

test_samples = test_gen_p2.samples[:limit]
y_true_sev = np.array([_get_sev(s) for s in test_samples])
y_true_off = np.array([_OFF_MAP.get(s.get('Offence', ''), 0) for s in test_samples])
y_true_act = np.array([_ACT_MAP.get(s.get('Action class', ''), 0) for s in test_samples])
y_pred_off_p2 = (pred_dict_p2['head_offence'][:limit].flatten() > 0.5).astype(int)
y_pred_act_p2 = np.argmax(pred_dict_p2['head_action'][:limit], axis=1)

results = {}
results['offence'] = print_raw_report(y_true_off, y_pred_off_p2, MAP_OFF,
                                      "HEAD OFFENCE (Pipeline 2)")
results['action'] = print_raw_report(y_true_act, y_pred_act_p2, MAP_ACT,
                                     "HEAD ACTION (Pipeline 2 — Macro Groups)")
results['severity'] = print_raw_report(y_true_sev, y_ens, MAP_SEV,
                                       "HEAD SEVERITY (Ensemble: P1 base + P2 RED)")

labels_sev = list(MAP_SEV.values())
cm = confusion_matrix(y_true_sev, y_ens)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=labels_sev, yticklabels=labels_sev)
plt.title('Confusion Matrix: Severity (Ensemble)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

os.makedirs(os.path.join(TRAINING_ROOT, 'models'), exist_ok=True)
results_path = os.path.join(TRAINING_ROOT, 'models', 'test_results_ensemble.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n  Results saved: {results_path}")